In [1]:
# job_posting_evaluator.py
import json
from xml.dom.expatbuilder import DOCUMENT_NODE

import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Any, Set
from dataclasses import dataclass
from collections import defaultdict
import re

from docutils.nodes import document
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import random
from pathlib import Path
import time


@dataclass
class JobQuery:
    """채용공고 검색 쿼리"""
    query: str
    expected_features: Dict[str, Any]  # 예상되는 결과의 특징들
    category: str
    query_type: str  # "keyword", "semantic", "filtered", "complex"


class JobPostingEvaluator:
    def __init__(self, json_file_path: str, retrievers: Dict[str, Any]):
        """
        json_file_path: 채용공고 JSON 파일 경로
        retrievers: {"retriever_name": retriever_instance, ...}
        """
        self.json_file_path = json_file_path
        self.retrievers = retrievers
        self.job_data = self.load_job_data()
        self.job_documents = self.create_documents()

    def load_job_data(self) -> Dict[str, Dict]:
        """채용공고 JSON 데이터 로드"""
        with open(self.json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"📁 {len(data)}개의 채용공고 데이터를 로드했습니다.")
        return data

    def create_documents(self) -> List[Dict]:
        """채용공고를 검색용 문서로 변환"""
        documents = []
        for job_id, job_info in self.job_data.items():
            # 텍스트 필드 결합
            text_content = []
            text_content.append(job_info.get("제목", ""))
            text_content.append(job_info.get("포지션 상세 설명", ""))

            # 주요 업무 처리
            if isinstance(job_info.get("주요 업무"), list):
                text_content.extend([task for task in job_info["주요 업무"] if task.strip()])

            # 자격 요건 처리
            if isinstance(job_info.get("자격 요건"), list):
                text_content.extend([req for req in job_info["자격 요건"] if req.strip()])

            # 우대 사항 처리
            if isinstance(job_info.get("우대 사항"), list):
                text_content.extend([pref for pref in job_info["우대 사항"] if pref.strip()])

            combined_text = " ".join(text_content)

            documents.append({
                "id": job_id,
                "content": combined_text,
                "metadata": job_info
            })

        return documents

    def generate_test_queries(self, num_queries: int = 100) -> List[JobQuery]:
        """실제 데이터 기반 테스트 쿼리 자동 생성"""
        queries = []

        # 1. 키워드 기반 쿼리 생성
        keyword_queries = self._generate_keyword_queries(num_queries // 4)
        queries.extend(keyword_queries)

        # 2. 의미적 쿼리 생성
        semantic_queries = self._generate_semantic_queries(num_queries // 4)
        queries.extend(semantic_queries)

        # 3. 필터링 쿼리 생성
        filtered_queries = self._generate_filtered_queries(num_queries // 4)
        queries.extend(filtered_queries)

        # 4. 복합 쿼리 생성
        complex_queries = self._generate_complex_queries(num_queries // 4)
        queries.extend(complex_queries)

        return queries

    def _generate_keyword_queries(self, num_queries: int) -> List[JobQuery]:
        """키워드 기반 쿼리 생성"""
        queries = []

        # 기술 스택 기반
        tech_keywords = ["Python", "Java", "React", "AWS", "머신러닝", "AI", "SQL", "Django", "Spring"]
        job_types = ["백엔드", "프론트엔드", "풀스택", "데이터", "마케팅", "기획", "영업", "디자인"]
        experience_levels = ["신입", "주니어", "시니어", "경력", "1년", "3년", "5년"]

        for _ in range(num_queries):
            tech = random.choice(tech_keywords)
            job_type = random.choice(job_types)
            exp = random.choice(experience_levels)

            query_templates = [
                f"{tech} {job_type} 개발자",
                f"{job_type} {exp} 채용",
                f"{tech} 개발자 {exp}",
                f"{job_type} 엔지니어"
            ]

            query = random.choice(query_templates)
            queries.append(JobQuery(
                query=query,
                expected_features={"기술스택": tech, "직무": job_type, "경력": exp},
                category="기술_키워드",
                query_type="keyword"
            ))

        return queries

    def _generate_semantic_queries(self, num_queries: int) -> List[JobQuery]:
        """의미적 쿼리 생성"""
        queries = []

        semantic_templates = [
            "성장하는 스타트업에서 일하고 싶어요",
            "원격근무가 가능한 회사를 찾고 있습니다",
            "데이터 분석 업무를 담당할 수 있는 포지션",
            "고객과 소통하는 업무를 선호합니다",
            "창의적인 프로젝트에 참여하고 싶습니다",
            "안정적인 대기업에서 근무하고 싶어요",
            "해외 진출 기회가 있는 회사",
            "팀워크를 중시하는 문화의 회사"
        ]

        for template in semantic_templates[:num_queries]:
            queries.append(JobQuery(
                query=template,
                expected_features={"쿼리타입": "의미적"},
                category="의미적_검색",
                query_type="semantic"
            ))

        return queries

    def _generate_filtered_queries(self, num_queries: int) -> List[JobQuery]:
        """필터링 조건 쿼리 생성"""
        queries = []

        locations = ["서울", "강남", "판교", "부산", "경기", "인천"]
        company_sizes = ["스타트업", "중소기업", "대기업", "외국계"]

        for _ in range(num_queries):
            location = random.choice(locations)
            company_size = random.choice(company_sizes)

            filter_templates = [
                f"{location} {company_size} 채용",
                f"{location}에서 근무할 수 있는 회사",
                f"{company_size} 개발자 모집",
                f"{location} 지역 IT 회사"
            ]

            query = random.choice(filter_templates)
            queries.append(JobQuery(
                query=query,
                expected_features={"지역": location, "회사규모": company_size},
                category="지역_회사규모",
                query_type="filtered"
            ))

        return queries

    def _generate_complex_queries(self, num_queries: int) -> List[JobQuery]:
        """복합 조건 쿼리 생성"""
        queries = []

        complex_templates = [
            "서울 스타트업 Python 백엔드 신입 개발자",
            "강남 대기업 마케팅 경력 3년 이상",
            "원격근무 가능한 React 프론트엔드 개발자",
            "판교 IT 회사 데이터 사이언티스트 주니어",
            "부산 지역 Java 개발자 시니어급",
            "외국계 회사 영업 매니저 경력직",
            "AI 머신러닝 엔지니어 석사 우대",
            "금융권 백엔드 개발자 5년 이상"
        ]

        for template in complex_templates[:num_queries]:
            queries.append(JobQuery(
                query=template,
                expected_features={"쿼리타입": "복합"},
                category="복합_조건",
                query_type="complex"
            ))

        return queries

    def calculate_relevance_score(self, query: JobQuery, retrieved_job: Dict) -> float:
        """검색된 채용공고의 관련도 점수 계산"""
        score = 0.0
        job_data = retrieved_job["metadata"]

        # 1. 키워드 매칭 점수 (40%)
        keyword_score = self._calculate_keyword_score(query.query, retrieved_job["content"])
        score += keyword_score * 0.4

        # 2. 카테고리 매칭 점수 (30%)
        category_score = self._calculate_category_score(query, job_data)
        score += category_score * 0.3

        # 3. 메타데이터 매칭 점수 (30%)
        metadata_score = self._calculate_metadata_score(query, job_data)
        score += metadata_score * 0.3

        return min(score, 1.0)

    def _calculate_keyword_score(self, query: str, content: str) -> float:
        """키워드 매칭 점수"""
        query_words = set(query.lower().split())
        content_words = set(content.lower().split())

        if not query_words:
            return 0.0

        intersection = query_words & content_words
        return len(intersection) / len(query_words)

    def _calculate_category_score(self, query: JobQuery, job_data: Dict) -> float:
        """카테고리 매칭 점수"""
        score = 0.0

        # 직군/직무 매칭
        job_title = job_data.get("제목", "").lower()
        job_group = job_data.get("직군", "").lower()

        query_lower = query.query.lower()

        # 개발자 관련
        if any(word in query_lower for word in ["개발자", "엔지니어", "백엔드", "프론트엔드"]):
            if any(word in job_title or word in job_group for word in ["개발", "엔지니어", "백엔드", "프론트"]):
                score += 0.5

        # 마케팅 관련
        if any(word in query_lower for word in ["마케팅", "광고", "홍보"]):
            if "마케팅" in job_group or "마케팅" in job_title:
                score += 0.5

        return score

    def _calculate_metadata_score(self, query: JobQuery, job_data: Dict) -> float:
        """메타데이터 매칭 점수"""
        score = 0.0
        query_lower = query.query.lower()

        # 지역 매칭
        location = job_data.get("지역", "").lower()
        if any(loc in query_lower for loc in ["서울", "강남", "판교", "부산"]):
            if any(loc in location for loc in ["서울", "강남", "판교", "부산"]):
                score += 0.3

        # 경력 매칭
        min_exp = job_data.get("요구 최소 경력", 0)
        if "신입" in query_lower and min_exp <= 1:
            score += 0.2
        elif "경력" in query_lower and min_exp > 1:
            score += 0.2

        # 기술 스택 매칭
        tech_stack = job_data.get("기술 스택", [])
        if tech_stack and isinstance(tech_stack, list):
            for tech in tech_stack:
                if tech.lower() in query_lower:
                    score += 0.3
                    break

        return min(score, 1.0)

    def evaluate_retriever(self, retriever_name: str, retriever: Any,
                          test_queries: List[JobQuery], k: int = 5) -> Dict[str, Any]:
        """단일 retriever 평가"""
        print(f"🔍 {retriever_name} 평가 중... (k={k})")

        results = {
            "precision_scores": [],
            "recall_scores": [],
            "ndcg_scores": [],
            "mrr_scores": [],
            "response_times": [],
            "category_performance": defaultdict(list)
        }

        for i, query in enumerate(test_queries):
            if i % 20 == 0:
                print(f"  진행률: {i}/{len(test_queries)}")

            start_time = time.time()

            try:
                # Retriever로 검색 수행
                retrieved_docs = retriever.get_relevant_documents(query.query)[:k]
                response_time = time.time() - start_time

                # 관련도 점수 계산
                relevance_scores = []
                for doc in retrieved_docs:
                    # Document 객체에서 메타데이터 추출
                    doc_id = doc.metadata.get("id", "")
                    if doc_id in self.job_data:
                        job_info = {
                            "content": doc.page_content,
                            "metadata": self.job_data[doc_id]
                        }
                        score = self.calculate_relevance_score(query, job_info)
                        relevance_scores.append(score)
                    else:
                        relevance_scores.append(0.0)

                # 메트릭 계산
                if relevance_scores:
                    # Precision@k (관련도 0.5 이상을 관련 문서로 간주)
                    relevant_count = sum(1 for score in relevance_scores if score >= 0.5)
                    precision = relevant_count / len(relevance_scores)

                    # MRR 계산
                    mrr = 0
                    for i, score in enumerate(relevance_scores):
                        if score >= 0.5:
                            mrr = 1 / (i + 1)
                            break

                    # NDCG 계산
                    dcg = sum(score / np.log2(i + 2) for i, score in enumerate(relevance_scores))
                    ideal_scores = sorted(relevance_scores, reverse=True)
                    idcg = sum(score / np.log2(i + 2) for i, score in enumerate(ideal_scores))
                    ndcg = dcg / idcg if idcg > 0 else 0

                    results["precision_scores"].append(precision)
                    results["mrr_scores"].append(mrr)
                    results["ndcg_scores"].append(ndcg)
                    results["response_times"].append(response_time)
                    results["category_performance"][query.category].append(precision)

            except Exception as e:
                print(f"  에러 발생 (쿼리: {query.query}): {e}")
                continue

        # 평균 계산
        final_results = {
            "avg_precision": np.mean(results["precision_scores"]) if results["precision_scores"] else 0,
            "avg_mrr": np.mean(results["mrr_scores"]) if results["mrr_scores"] else 0,
            "avg_ndcg": np.mean(results["ndcg_scores"]) if results["ndcg_scores"] else 0,
            "avg_response_time": np.mean(results["response_times"]) if results["response_times"] else 0,
            "category_performance": {
                cat: np.mean(scores) for cat, scores in results["category_performance"].items()
            },
            "total_queries": len(results["precision_scores"])
        }

        return final_results

    def evaluate_all_retrievers(self, test_queries: List[JobQuery], k: int = 5) -> Dict[str, Dict]:
        """모든 retriever 평가"""
        all_results = {}

        for retriever_name, retriever in self.retrievers.items():
            results = self.evaluate_retriever(retriever_name, retriever, test_queries, k)
            all_results[retriever_name] = results

        return all_results

    def print_comparison(self, results: Dict[str, Dict]):
        """결과 비교 출력"""
        print("\n" + "="*80)
        print("📊 Retriever 성능 비교 결과")
        print("="*80)

        # 헤더
        print(f"{'Retriever':<20} {'Precision':<12} {'MRR':<12} {'NDCG':<12} {'Response(s)':<12} {'Queries':<8}")
        print("-" * 80)

        # 각 retriever 결과
        for name, result in results.items():
            print(f"{name:<20} {result['avg_precision']:<12.4f} {result['avg_mrr']:<12.4f} "
                  f"{result['avg_ndcg']:<12.4f} {result['avg_response_time']:<12.4f} {result['total_queries']:<8}")

        # 카테고리별 성능
        print("\n📈 카테고리별 성능 (Precision)")
        print("-" * 60)

        all_categories = set()
        for result in results.values():
            all_categories.update(result['category_performance'].keys())

        print(f"{'Category':<20}", end="")
        for name in results.keys():
            print(f"{name:<15}", end="")
        print()
        print("-" * 60)

        for category in sorted(all_categories):
            print(f"{category:<20}", end="")
            for name, result in results.items():
                score = result['category_performance'].get(category, 0)
                print(f"{score:<15.4f}", end="")
            print()

    def save_results(self, results: Dict[str, Dict], filename: str = "job_retriever_evaluation.json"):
        """결과 저장"""
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"\n💾 평가 결과가 {filename}에 저장되었습니다.")


# 사용 예시 함수
def run_job_posting_evaluation(json_file_path: str, retriever_builder):
    """채용공고 데이터로 retriever 평가 실행"""

    # 1. 다양한 retriever 생성
    retrievers = {
        "Vector_Only": retriever_builder.build_with_mode(1),
        "BM25_Only": retriever_builder.build_with_mode(2),
        "Ensemble": retriever_builder.build_with_mode(3),
        "Parent_Document": retriever_builder.build_with_mode(4)
    }

    # 2. 평가자 초기화
    evaluator = JobPostingEvaluator(json_file_path, retrievers)

    # 3. 테스트 쿼리 생성
    test_queries = evaluator.generate_test_queries(num_queries=80)
    print(f"📝 {len(test_queries)}개의 테스트 쿼리를 생성했습니다.")

    # 4. 평가 실행
    results = evaluator.evaluate_all_retrievers(test_queries, k=5)

    # 5. 결과 출력 및 저장
    evaluator.print_comparison(results)
    evaluator.save_results(results)

    return results, evaluator


# 추가: 실시간 쿼리 테스트 함수
def test_live_queries(evaluator: JobPostingEvaluator, retriever_name: str):
    """실시간으로 쿼리 테스트"""
    retriever = evaluator.retrievers[retriever_name]

    print(f"\n🔍 {retriever_name} 실시간 테스트")
    print("종료하려면 'quit'를 입력하세요.")

    while True:
        query = input("\n검색 쿼리: ").strip()
        if query.lower() == 'quit':
            break

        if not query:
            continue

        try:
            start_time = time.time()
            docs = retriever.get_relevant_documents(query)[:5]
            response_time = time.time() - start_time

            print(f"\n📊 검색 결과 ({response_time:.3f}초):")
            print("-" * 50)

            for i, doc in enumerate(docs, 1):
                job_id = doc.metadata.get("id", "Unknown")
                if job_id in evaluator.job_data:
                    job_info = evaluator.job_data[job_id]
                    print(f"{i}. {job_info.get('제목', 'No Title')}")
                    print(f"   회사: {job_info.get('회사명', 'Unknown')}")
                    print(f"   지역: {job_info.get('근무지', '')} {job_info.get('지역', '')}")
                    print(f"   직군: {job_info.get('직군', 'Unknown')}")
                    print()

        except Exception as e:
            print(f"❌ 오류: {e}")


if __name__ == "__main__":
    # 실행 예시
    json_file_path = "Data_Files/wanted_detail_improve_20250616.json"

    # RetrieverBuilder는 별도로 초기화 필요
    # results, evaluator = run_job_posting_evaluation(json_file_path, retriever_builder)

    # 실시간 테스트
    # test_live_queries(evaluator, "Ensemble")

In [2]:
# job_data_processor.py
import json
import re
from typing import List, Dict, Any
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
from collections import Counter


class JobDataProcessor:
    def __init__(self, json_file_path: str):
        self.json_file_path = json_file_path
        self.job_data = self.load_data()

    def load_data(self) -> Dict[str, Dict]:
        """JSON 파일에서 채용공고 데이터 로드"""
        with open(self.json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"📁 {len(data)}개의 채용공고를 로드했습니다.")
        return data

    def clean_text(self, text: str) -> str:
        """텍스트 전처리"""
        if not isinstance(text, str):
            return ""

        # 불필요한 공백 제거
        text = re.sub(r'\s+', ' ', text)
        # 특수문자 정리 (단, 중요한 기호는 유지)
        text = re.sub(r'[^\w\s가-힣.,()/-]', '', text)
        return text.strip()

    def process_list_field(self, field_data: Any) -> str:
        """리스트 형태의 필드를 텍스트로 변환"""
        if not field_data:
            return ""

        if isinstance(field_data, list):
            # 빈 문자열 제거 후 조인
            clean_items = [self.clean_text(str(item)) for item in field_data if str(item).strip()]
            return " ".join(clean_items)
        else:
            return self.clean_text(str(field_data))

    def create_full_documents(self) -> List[Document]:
        """전체 채용공고를 Document 객체로 변환"""
        documents = []

        for job_id, job_info in self.job_data.items():
            # 각 필드별로 텍스트 생성
            content_parts = []

            # 제목 (가중치 높음)
            title = job_info.get("제목", "")
            if title:
                content_parts.append(f"제목: {self.clean_text(title)}")

            # 회사명
            company = job_info.get("회사명", "")
            if company:
                content_parts.append(f"회사: {self.clean_text(company)}")

            # 포지션 상세 설명
            description = job_info.get("포지션 상세 설명", "")
            if description:
                content_parts.append(f"설명: {self.clean_text(description)}")

            # 주요 업무
            tasks = self.process_list_field(job_info.get("주요 업무", []))
            if tasks:
                content_parts.append(f"주요업무: {tasks}")

            # 자격 요건
            requirements = self.process_list_field(job_info.get("자격 요건", []))
            if requirements:
                content_parts.append(f"자격요건: {requirements}")

            # 우대 사항
            preferences = self.process_list_field(job_info.get("우대 사항", []))
            if preferences:
                content_parts.append(f"우대사항: {preferences}")

            # 기술 스택
            tech_stack = self.process_list_field(job_info.get("기술 스택", []))
            if tech_stack:
                content_parts.append(f"기술스택: {tech_stack}")

            # 직군/직무
            job_group = job_info.get("직군", "")
            if job_group:
                content_parts.append(f"직군: {self.clean_text(job_group)}")

            job_roles = self.process_list_field(job_info.get("직무", []))
            if job_roles:
                content_parts.append(f"직무: {job_roles}")

            # 지역 정보
            location = job_info.get("근무지", "")
            region = job_info.get("지역", "")
            if location or region:
                content_parts.append(f"지역: {self.clean_text(location)} {self.clean_text(region)}")

            # 경력 정보
            min_exp = job_info.get("요구 최소 경력", "")
            if min_exp:
                content_parts.append(f"경력: {min_exp}년")

            # 신입 가능 여부
            is_entry = job_info.get("신입 지원 가능 여부", False)
            if is_entry:
                content_parts.append("신입가능")

            # 복지
            benefits = self.process_list_field(job_info.get("복지", []))
            if benefits:
                content_parts.append(f"복지: {benefits}")

            # 회사 특징
            features = self.process_list_field(job_info.get("특징", []))
            if features:
                content_parts.append(f"특징: {features}")

            # 전체 텍스트 결합
            full_content = " ".join(content_parts)

            # Document 객체 생성
            document = Document(
                page_content=full_content,
                metadata={
                    "id": job_id,
                    "title": title,
                    "company": company,
                    "location": f"{location} {region}".strip(),
                    "job_group": job_group,
                    "min_experience": min_exp,
                    "is_entry_level": is_entry,
                    "tech_stack": job_info.get("기술 스택", []),
                    "job_roles": job_info.get("직무", []),
                    "source": "wanted"
                }
            )


In [3]:
document[0]

NameError: name 'document' is not defined